# Setup

## Environment

In [ ]:
# %%bash
# 1. Parameterize target branch name
BRANCH="LLM"
MODEL = "SciDFM"

# 2. Reset workspace directory for a clean slate
!rm -rf FoSpy

# 3. Fast clone single branch
!git clone --single-branch --branch $BRANCH https://github.com/errthumt/FoSpy.git FoSpy

# 4. Change notebook working directory to target folder
%cd FoSpy/LLM/sandbox

# 5. Fast dependency installation using uv without touching native PyTorch
%pip install -q uv
!uv pip install -r colab_reqs.txt #--no-deps
#!uv pip install -q safetensors fsspec huggingface_hub


## Python Imports

In [ ]:
import json
import os
import re
from pathlib import Path
import torch
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    GenerationConfig,
    LlamaTokenizer,
)


## Globals and Helpers

In [ ]:
INPUT_SOURCES = {
    "Ba2Zn5As6 Supplemental Information": (
        "Ba2Zn5As6_SI.txt",
        "Solid-State Material Science",
    )
}


# Define default path pointers
INPUT_DIR = Path("inputs")
OUTPUT_FILE = Path(MODEL) / "output" / "tokens.json"

# Define fallback evaluation arguments
MAX_TOKENS = 2048
TEMPERATURE = 0.1
MODEL_ID = "OpenDFM/SciDFM-MoE-A5.6B-v1.0"

# --- BRANCH FLOW CONTROLS ---
# Default local fallbacks (will be updated if Drive cell runs)
CACHE_DIR = "/content/SciDFM_Cache/local"
LOCAL_FILES_ONLY = False

# Define quantization configurations once globally
QUANT_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# Global Generation Settings
GEN_CONFIG = GenerationConfig(
    do_sample=True,
    top_k=20,
    top_p=0.9,
    temperature=TEMPERATURE,
    max_new_tokens=MAX_TOKENS,
    eos_token_id=1,  # Set dynamic fallback ID or override downstream
)


def get_prompt(input_text, context="None Specified"):
    return f"""
    Extract the fundamental scientific information pertaining to the synthetic
    method and product characterization described in the following input. Return
    the result as a JSON-formatted list of atomic, order-independent scientific information
    tokens. Each token must represent one fact, relationship or entity. Whenever
    possible, token values should be coercible to primitive data types,
    including separating quantities into a `value` and `units` fields. If
    specified, context refers to the scientific field or chemical category
    pertaining to the intended synthetic products.

    Rules:
    - Remove redundancy.
    - Normalize chemical formulas.
    - Normalize references to individual elements (outside chemical formulas) to their full element name.
    - Each token must be independent of sentence order.
    - Token meaning must be independent of narrative context.
    - A preamble before JSON output is optional, but must not contain any JSON-signaling characters such as {{ or [.
    - No further response characters are allowed after the JSON output.

    Input:
    \"\"\"
    {input_text}
    \"\"\"

    Context:
    \"\"\"
    {context}
    \"\"\"
    """


# Get Model

## (Optional) Google Drive Cache

Skip this cell if you want to download directly from Hugging Face. Run this cell only if you want to switch the execution configuration to load weights locally from your persistent Drive cloud backup.

In [ ]:
from google.colab import drive

print("Activating Google Drive branching configuration...")

# 1. Mount physical personal Drive layout
drive.mount("/content/drive")

# 2. Pivot default variable markers to target drive targets
CACHE_DIR = "/content/drive/MyDrive/SciDFM_Cache"


print(f"Branch changed! Model will now search for weights in: {CACHE_DIR}")


### Switch to Local Files

Only use this cell if a Google Drive Cache has already been built. For the first drive cache build, disable it.

In [ ]:
LOCAL_FILES_ONLY = True

## (Required) Build Model
This cell either builds the model fresh, or, if the Google Drive Cache was setup, builds the model in/out of the new cache dir

In [ ]:
print(f"Initializing Tokenizer from target: {CACHE_DIR}")
tokenizer = LlamaTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=False,
    trust_remote_code=True,
    cache_dir=CACHE_DIR,
    local_files_only=LOCAL_FILES_ONLY,
)

# Bind generation configurations to the specific token profile dynamically
GEN_CONFIG.eos_token_id = tokenizer.eos_token_id

print("Loading Model layers (4-bit quantization active)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=QUANT_CONFIG,
    device_map="auto",
    trust_remote_code=True,
    cache_dir=CACHE_DIR,
    local_files_only=LOCAL_FILES_ONLY,
)
print("Model configuration loaded and ready for execution!")


# Main Logic

In [ ]:
# Ensure the workspace directory physically exists
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

out = {"tokens": {}, "full_responses": {}}
out_tokens = out["tokens"]
out_full = out["full_responses"]

for name, (file, context) in INPUT_SOURCES.items():
    target_file = INPUT_DIR / file
    if not target_file.exists():
        print(f"ERROR: Target input file missing at path: {target_file}")
        continue

    input_text = target_file.read_text()
    prompt = get_prompt(input_text, context)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, generation_config=GEN_CONFIG)

    raw = tokenizer.decode(outputs[0], skip_special_tokens=True)[len(prompt) :]
    out_full[name] = raw

    matched = re.search(r"[\{\[].*[\}\]]", raw, flags=re.DOTALL)
    if not matched:
        print(f"ERROR: No JSON array or object pattern isolated for {name}")
        continue

    json_str = matched.group(0)
    try:
        out_tokens[name] = json.loads(json_str)
    except json.JSONDecodeError:
        print(f"ERROR: Truncated or invalid JSON block isolated: {json_str}")

with OUTPUT_FILE.open("w") as f:
    json.dump(out, f, indent=4)

print(f"Pipeline executed successfully. Extraction matrix written to: {OUTPUT_FILE}")
